# Active_Auth — Đánh giá đầy đủ & Kiểm chứng phương pháp

**Vai trò:** chạy TOÀN BỘ phục vụ phân tích trong báo cáo. Bao trùm mọi thứ của `Active_Auth_Train_Deploy.ipynb`, CỘNG THÊM:

- So sánh 4 hàm chấm điểm (cos_mean / cos_knn / cos_znorm / maha) — nguồn **Bảng 5.5**: in bảng "OWNER VERIFICATION"; chạy nhiều cụm `HELD_OUT_IDS` rồi tổng hợp trung bình ± độ lệch chuẩn.
- Phát hiện người lạ (membership OOD: msp / energy / openmax).

> Dùng khi cần tái lập số liệu đánh giá/benchmark. Chỉ huấn luyện + xuất bản triển khai thì dùng `Active_Auth_Train_Deploy.ipynb`.


# Active Authentication — Notebook CHẠY 1 LƯỢT (Run all)

**Cách dùng:** sửa khối **CONFIG** ở cell dưới rồi bấm *Runtime → Run all*.
Notebook tự: mount Drive → ghi module → train backbone (mỗi context mode) → đánh giá open/closed-set → **kiểm chứng 7 phương pháp scoring** (§ cuối).

Không cần file ngoài. Mỗi module là một cell `%%writefile` để bạn xem/sửa được.

## 1. CONFIG — chỉnh ở đây

In [ ]:
DATA_DIR      = "/content/drive/MyDrive/DATN/processed"
HELD_OUT_IDS  = ["user22", "user23", "user24", "user25", "user26"]
CONTEXT_MODES = ["walking", "all"]
N_EVAL_RUNS   = 5
RUN_BENCHMARK = True
PACKAGE_ZIP   = True

## 2. Setup — mount Drive, cài thư viện, kiểm tra GPU & dữ liệu

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Đã mount /content/drive")
except Exception:
    print("Không phải Colab — bỏ qua mount.")

import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "scikit-learn", "pandas", "numpy", "scipy"], check=False)
import torch, sklearn, numpy, pandas
print("torch:", torch.__version__, " CUDA:", torch.cuda.is_available())

from pathlib import Path
assert Path(DATA_DIR).exists(), f"Không tìm thấy DATA_DIR: {DATA_DIR}"
_users = sorted(d.name for d in Path(DATA_DIR).iterdir() if d.is_dir())
_held  = [u for u in HELD_OUT_IDS if u in _users]
print(f"Tổng user: {len(_users)} | HELD-OUT có mặt: {len(_held)} {_held}")
assert len(_held) >= 1, "Held-out không có trong DATA_DIR — đã upload đủ user chưa?"

## 3. Ghi toàn bộ module pipeline ra đĩa

Mỗi cell ghi một file. Đây là mã gốc của pipeline, giữ nguyên.

In [ ]:
%%writefile config.py
"""
config.py — Pipeline v4 + CONTEXT_MODE switch

CONTEXT_MODE quy định backbone được train với loại data nào:
  - 'walking' : chỉ walking windows (X_walking.npy) — gait pattern thuần
  - 'all'     : cả 3 activity (X_inertial.npy) — walking + sitting + standing

Output dir tự suffix theo mode để 2 lần train không đè lên nhau.
Mode đọc từ env var CONTEXT_MODE (do main.py set qua CLI --context).
"""
import os
from pathlib import Path

CONTEXT_MODE = os.environ.get('CONTEXT_MODE', 'all')
assert CONTEXT_MODE in ('walking', 'all'),    f"CONTEXT_MODE phải là 'walking' hoặc 'all', không phải {CONTEXT_MODE!r}"

DATA_DIR    = Path("./processed_data")
MODELS_DIR  = Path(f"./models_{CONTEXT_MODE}")
RESULTS_DIR = Path(f"./results_{CONTEXT_MODE}")
EXPORT_DIR  = Path(f"./export_{CONTEXT_MODE}")

SENSOR_COLS = [
    "acc_x", "acc_y", "acc_z",
    "gyro_x", "gyro_y", "gyro_z",
    "mag_x",  "mag_y",  "mag_z",
]
WINDOW_SIZE = 200
N_CHANNELS  = 9
TOUCH_DIM   = 48

TEST_SIZE   = 0.20
VAL_SIZE    = 0.125

BACKBONE_ARCH     = 'cnn'
EMBED_DIM         = 128
BACKBONE_EPOCHS   = 30
BACKBONE_LR       = 1e-3
BACKBONE_BATCH    = 256
BACKBONE_PATIENCE = 8

POOL_SIZE_INERTIAL = 100
POOL_SIZE_TOUCH    = 100

RF_N_ESTIMATORS              = 200
RF_MAX_FEATURES              = "sqrt"
RF_CLASS_WEIGHT              = "balanced"
RF_MIN_SAMPLES_LEAF_INERTIAL = 2
RF_MIN_SAMPLES_LEAF_TOUCH    = 1

RETRY_MAX_ATTEMPTS = 2
CATASTROPHIC_AUC_T = 0.55
CATASTROPHIC_FRR   = 0.50
RETRY_SEED_OFFSET  = 7919

SEED   = 42

AUG_NOISE_STD    = 0.02
AUG_SCALE_RANGE  = (0.95, 1.05)
BALANCE_STRATEGY = "augment"

DATA_LOADER_WORKERS = 0
PIN_MEMORY          = True
PRELOAD_TO_DEVICE   = True
USE_AMP             = True
USE_ONECYCLE        = True
USE_FUSED_ADAM      = True
ENABLE_TF32         = True
LR_FACTOR           = 0.5
LR_PATIENCE         = 5
MIN_LR              = 1e-5

FUSION_STEPS         = 51
FUSION_TIE_TOLERANCE = 1e-6

NEG_RATIO = 1.0

In [ ]:
%%writefile dataset.py
"""
dataset.py — Load data per user + build session-aware train/val/test splits.

Hàm chính:
  load_users(data_dir, exclude_users, context_mode)
      Đọc tất cả user-folder trong data_dir, trả {uid: {"X", "session"}}.
      context_mode = 'walking' | 'all' quyết định load file nào.

  build_dataset_per_split(users, owner_id, ...)
      Per-impostor session split: owner và TỪNG impostor đều có session
      riêng cho train/val/test → không có user nào bị "rò" qua các split.
      Train set có augmentation (noise + scaling) để cân bằng số positive
      với số negative.

  prepare_tensors(X, y, device)
      Numpy → torch tensor channels-first (N, 9, WINDOW_SIZE), optional
      preload thẳng lên GPU để tránh host→device copy mỗi batch.
"""

import numpy as np
import torch
from torch.utils.data import Dataset
from pathlib import Path

from config import (
    SENSOR_COLS, WINDOW_SIZE, N_CHANNELS, TOUCH_DIM,
    NEG_RATIO, AUG_NOISE_STD, AUG_SCALE_RANGE,
    BALANCE_STRATEGY, CONTEXT_MODE,
)

def load_users(data_dir: Path,
               exclude_users: set | None = None,
               context_mode: str | None = None) -> dict:
    """
    Load tat ca user tu data_dir/.

    context_mode (None → lấy từ config.CONTEXT_MODE):
      - 'walking' : X_walking.npy + y_walking.npy   (chỉ windows từ walking)
      - 'all'     : X_inertial.npy + y_inertial.npy (walking + sitting + standing)

    y_*.npy chứa prefixed session strings (ví dụ: 'user1_session_1').
    """
    if exclude_users is None:
        exclude_users = set()

    if context_mode is None:
        context_mode = CONTEXT_MODE
    if context_mode not in ('walking', 'all'):
        raise ValueError(f"context_mode phải là 'walking' hoặc 'all', không phải {context_mode!r}")

    if context_mode == 'walking':
        x_name, y_name = "X_walking.npy",  "y_walking.npy"
    else:
        x_name, y_name = "X_inertial.npy", "y_inertial.npy"

    print(f"  [load_users] context_mode = {context_mode!r}  →  load {x_name}")

    users = {}
    for user_dir in sorted(data_dir.iterdir()):
        if not user_dir.is_dir():
            continue
        if user_dir.name in exclude_users:
            print(f"  [skip] {user_dir.name}: excluded")
            continue

        x_path = user_dir / x_name
        y_path = user_dir / y_name

        if not (x_path.exists() and y_path.exists()):
            print(f"  Warning: Bỏ qua {user_dir.name}: thiếu {x_name}/{y_name}")
            continue

        X    = np.load(x_path).astype(np.float32)
        sess = np.load(y_path, allow_pickle=True)

        if X.shape[1] != WINDOW_SIZE or X.shape[2] != N_CHANNELS:
            print(f"  Warning: {user_dir.name}: X shape {X.shape} != (N,{WINDOW_SIZE},{N_CHANNELS})")
            continue
        if len(X) != len(sess):
            print(f"  Warning: {user_dir.name}: X({len(X)}) != sess({len(sess)})")
            continue

        n_sess = len(np.unique(sess))
        users[user_dir.name] = {"X": X, "session": sess}
        print(f"  Loaded {user_dir.name}: {len(X):,} windows, {n_sess} sessions")

    return users
def build_dataset_single_user(user_data: dict, seed: int = 42) -> tuple:
    """
    [UNSAFE] Single-user mode: synthesize impostors bằng noise.

    KHÔNG có session-aware split. KHÔNG được dùng cho evaluation thật.
    Pipeline mới sẽ raise thay vì gọi fallback này.
    """
    rng  = np.random.default_rng(seed)
    X    = user_data["X"]
    sess = user_data["session"]

    pos_X    = X.copy()
    pos_y    = np.ones(len(pos_X), dtype=np.float32)
    pos_sess = sess.copy()

    n_neg    = len(pos_X)
    idx      = rng.integers(0, len(X), size=n_neg)
    neg_X    = X[idx].copy()
    neg_X   += rng.normal(0, 0.5, neg_X.shape).astype(np.float32)
    neg_X   *= rng.uniform(0.7, 1.3, size=(n_neg, 1, 1)).astype(np.float32)
    neg_y    = np.zeros(n_neg, dtype=np.float32)
    neg_sess = sess[idx]

    X_all    = np.concatenate([pos_X, neg_X])
    y_all    = np.concatenate([pos_y, neg_y])
    sess_all = np.concatenate([pos_sess, neg_sess])
    perm     = rng.permutation(len(X_all))

    print("    ⚠⚠⚠ Single-user mode: KHÔNG session-aware, kết quả không tin cậy")
    return X_all[perm].astype(np.float32), y_all[perm], sess_all[perm]

def build_dataset_per_split(users: dict,
                              owner_id: str,
                              test_size: float = 0.20,
                              val_size: float  = 0.125,
                              neg_ratio: float = NEG_RATIO,
                              seed: int = 42,
                              strategy: str | None = None) -> dict:
    """
    Build dataset theo per-impostor session split.

    Quy trình (BUG-6 fix):
      - Owner: split sessions → own_train / own_val / own_test
      - Mỗi impostor: split sessions riêng lẻ → imp_train / imp_val / imp_test
      - Gộp cùng nhãn → final train/val/test
      → Không có người impostor nào xuất hiện cùng session ở cả train lẫn test

    Returns
    -------
    dict với keys:
        X_tr, y_tr, X_val, y_val, X_te, y_te  — numpy arrays
        sess_val, sess_te                       — session ids (PREFIXED)
    """
    if strategy is None:
        strategy = BALANCE_STRATEGY

    rng   = np.random.default_rng(seed)
    owner = users[owner_id]

    owner_sessions = np.unique(owner["session"])

    if len(owner_sessions) < 3:
        raise ValueError(
            f"Owner '{owner_id}' chỉ có {len(owner_sessions)} session "
            f"(cần ≥ 3 để chia train/val/test)."
        )

    rng.shuffle(owner_sessions)

    n_own_te  = max(1, int(len(owner_sessions) * test_size))
    n_own_val = max(1, int((len(owner_sessions) - n_own_te) * val_size))

    own_te_sess  = owner_sessions[:n_own_te]
    own_rem      = owner_sessions[n_own_te:]
    own_val_sess = own_rem[:n_own_val]
    own_tr_sess  = own_rem[n_own_val:]

    if len(own_tr_sess) == 0:
        raise ValueError(
            f"Owner '{owner_id}': sau khi chia, train có 0 session."
        )

    def _idx_by_sess(X, sess, target_sessions):
        mask = np.isin(sess, target_sessions)
        return X[mask], sess[mask]

    own_X_tr,  own_s_tr  = _idx_by_sess(owner["X"], owner["session"], own_tr_sess)
    own_X_val, own_s_val = _idx_by_sess(owner["X"], owner["session"], own_val_sess)
    own_X_te,  own_s_te  = _idx_by_sess(owner["X"], owner["session"], own_te_sess)

    others = [u for u in users if u != owner_id]

    imp_X_tr,  imp_s_tr  = [], []
    imp_X_val, imp_s_val = [], []
    imp_X_te,  imp_s_te  = [], []

    skipped = []

    for uid in others:
        data   = users[uid]
        u_sess = np.unique(data["session"])

        if len(u_sess) < 2:
            print(f"  ⚠ {uid}: chỉ có {len(u_sess)} session → chỉ vào train, "
                  f"KHÔNG xuất hiện ở val/test")
            skipped.append(uid)
            imp_X_tr.append(data["X"])
            imp_s_tr.append(data["session"])
            continue

        rng.shuffle(u_sess)
        n_u_te  = max(1, int(len(u_sess) * test_size))
        n_u_val = max(1, int((len(u_sess) - n_u_te) * val_size))

        u_te_sess  = u_sess[:n_u_te]
        u_rem      = u_sess[n_u_te:]
        u_val_sess = u_rem[:n_u_val] if n_u_val > 0 else np.array([], dtype=u_rem.dtype)
        u_tr_sess  = u_rem[n_u_val:]

        uX_tr,  us_tr  = _idx_by_sess(data["X"], data["session"], u_tr_sess)
        uX_val, us_val = _idx_by_sess(data["X"], data["session"], u_val_sess)
        uX_te,  us_te  = _idx_by_sess(data["X"], data["session"], u_te_sess)

        if len(uX_tr)  > 0: imp_X_tr.append(uX_tr);   imp_s_tr.append(us_tr)
        if len(uX_val) > 0: imp_X_val.append(uX_val); imp_s_val.append(us_val)
        if len(uX_te)  > 0: imp_X_te.append(uX_te);   imp_s_te.append(us_te)

    if skipped:
        print(f"  ℹ {len(skipped)} impostor không vào val/test "
              f"(test set không kiểm tra reject họ): {skipped}")

    if not imp_X_tr or not imp_X_val or not imp_X_te:
        raise ValueError(
            f"Không đủ impostor cho per-impostor split của owner '{owner_id}' "
            f"(train_empty={not imp_X_tr}, val_empty={not imp_X_val}, "
            f"test_empty={not imp_X_te})."
        )

    def _concat_split(pos_X, pos_s, neg_X_list, neg_s_list, do_augment=False):
        pos_y = np.ones(len(pos_X), dtype=np.float32)
        if not neg_X_list:
            return pos_X, pos_y, pos_s
        neg_X = np.concatenate(neg_X_list)
        neg_y = np.zeros(len(neg_X), dtype=np.float32)
        neg_s = np.concatenate(neg_s_list)

        if do_augment and strategy == "augment" and len(pos_X) < len(neg_X):
            n_needed = len(neg_X) - len(pos_X)
            aug      = _augment_windows(pos_X, n_needed, rng)
            aug_s    = rng.choice(pos_s, size=n_needed)
            pos_X    = np.concatenate([pos_X, aug])
            pos_y    = np.ones(len(pos_X), dtype=np.float32)
            pos_s    = np.concatenate([pos_s, aug_s])

        X = np.concatenate([pos_X, neg_X])
        y = np.concatenate([pos_y, neg_y])
        s = np.concatenate([pos_s, neg_s])
        perm = rng.permutation(len(X))
        return X[perm], y[perm], s[perm]

    X_tr,  y_tr,  s_tr  = _concat_split(own_X_tr,  own_s_tr,
                                          imp_X_tr,  imp_s_tr,  do_augment=True)
    X_val, y_val, s_val = _concat_split(own_X_val, own_s_val,
                                          imp_X_val, imp_s_val, do_augment=False)
    X_te,  y_te,  s_te  = _concat_split(own_X_te,  own_s_te,
                                          imp_X_te,  imp_s_te,  do_augment=False)

    for name, yy in [("val", y_val), ("test", y_te)]:
        if len(np.unique(yy)) < 2:
            print(f"  ⚠ {name} set chỉ có 1 class → AUC/EER không tin cậy")

    return dict(
        X_tr=X_tr.astype(np.float32),   y_tr=y_tr,
        X_val=X_val.astype(np.float32), y_val=y_val,
        X_te=X_te.astype(np.float32),   y_te=y_te,
        sess_val=s_val,
        sess_te=s_te,
    )

def build_dataset(users: dict,
                  owner_id: str,
                  neg_ratio: float = NEG_RATIO,
                  seed: int = 42,
                  strategy: str | None = None) -> tuple:
    """[LEGACY] Pipeline cũ — KHÔNG session-aware."""
    if strategy is None:
        strategy = BALANCE_STRATEGY

    rng   = np.random.default_rng(seed)
    owner = users[owner_id]

    pos_X    = owner["X"].copy()
    pos_y    = np.ones(len(pos_X), dtype=np.float32)
    pos_sess = owner["session"].copy()

    others       = [u for u in users if u != owner_id]
    n_neg_target = int(len(pos_X) * neg_ratio)

    neg_X_list, neg_sess_list = [], []
    collected = 0
    for uid in rng.permutation(others):
        if collected >= n_neg_target:
            break
        data   = users[uid]
        remain = n_neg_target - collected
        take   = min(len(data["X"]), remain)
        idx    = rng.choice(len(data["X"]), size=take, replace=False)
        neg_X_list.append(data["X"][idx])
        neg_sess_list.append(data["session"][idx])
        collected += take

    if not neg_X_list:
        return build_dataset_single_user(users[owner_id], seed=seed)

    neg_X    = np.concatenate(neg_X_list)
    neg_y    = np.zeros(len(neg_X), dtype=np.float32)
    neg_sess = np.concatenate(neg_sess_list)

    if strategy == "augment" and len(pos_X) < len(neg_X):
        n_needed = len(neg_X) - len(pos_X)
        aug      = _augment_windows(pos_X, n_needed, rng)
        aug_sess = rng.choice(pos_sess, size=n_needed)
        pos_X    = np.concatenate([pos_X, aug])
        pos_y    = np.ones(len(pos_X), dtype=np.float32)
        pos_sess = np.concatenate([pos_sess, aug_sess])

    X    = np.concatenate([pos_X,    neg_X])
    y    = np.concatenate([pos_y,    neg_y])
    sess = np.concatenate([pos_sess, neg_sess])
    perm = rng.permutation(len(X))
    return X[perm].astype(np.float32), y[perm], sess[perm]

def _augment_windows(windows: np.ndarray,
                     n_needed: int,
                     rng: np.random.Generator) -> np.ndarray:
    idx     = rng.integers(0, len(windows), size=n_needed)
    samples = windows[idx].copy()
    samples += rng.normal(0, AUG_NOISE_STD, samples.shape).astype(np.float32)
    scales  = rng.uniform(*AUG_SCALE_RANGE, size=(n_needed, 1, 1)).astype(np.float32)
    return samples * scales

def prepare_tensors(X: np.ndarray,
                    y: np.ndarray,
                    device: torch.device | None = None) -> tuple:
    """
    Convert numpy (N, WINDOW_SIZE, 9) → tensor (N, 9, WINDOW_SIZE) channels-first.
    Nếu device được cung cấp → move sang device luôn (eliminates host→device
    transfer mỗi batch trong training loop).

    Returns
    -------
    (X_tensor, y_tensor) — torch.float32, contiguous
    """
    X_t = torch.from_numpy(np.ascontiguousarray(X.transpose(0, 2, 1))).float()
    y_t = torch.from_numpy(y).float()
    if device is not None:
        X_t = X_t.to(device, non_blocking=False)
        y_t = y_t.to(device, non_blocking=False)
    return X_t, y_t

class InertialDataset(Dataset):
    """
    Wrap (X, y) numpy arrays thành PyTorch Dataset.
    Dùng khi PRELOAD_TO_DEVICE = False hoặc chạy CPU-only.
    X: (N, WINDOW_SIZE, 9)  →  lưu dạng (N, 9, WINDOW_SIZE) cho Conv1d channels-first
    """

    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.from_numpy(
            np.ascontiguousarray(X.transpose(0, 2, 1))
        ).float()
        self.y = torch.from_numpy(y).float()

    def __len__(self) -> int:
        return len(self.X)

    def __getitem__(self, idx: int):
        return self.X[idx], self.y[idx]

In [ ]:
%%writefile touch_features.py
"""
touch_features.py — 48-D touch feature schema.

Vector 48 chiều (thứ tự cố định):
  TAP    (16): tap_n, tap_hold x5, tap_disp x5, tap_iti x5
  SCROLL (23): scroll_n, dur x2, traj x2, sdist x2, vmean x2,
               vmax x2, vlast5 x2, mrl x2, afirst5 x2,
               dir_circ x2, frac x4
  KEY     (9): key_n, key_inter x5, delete_rate, typing_speed, burst_rate

API:
  build_session_features(user_dir, session_ids) -> np.ndarray (48,) | None
  strip_user_prefix(prefixed_session, user_id) -> str
"""

import numpy as np
import pandas as pd
from pathlib import Path

TAP_COLS = [
    "tap_n",
    "tap_hold_mean",   "tap_hold_std",   "tap_hold_median", "tap_hold_p25",   "tap_hold_p75",
    "tap_disp_mean",   "tap_disp_std",   "tap_disp_median", "tap_disp_p25",   "tap_disp_p75",
    "tap_iti_mean",    "tap_iti_std",    "tap_iti_median",  "tap_iti_p25",    "tap_iti_p75",
]
SCROLL_COLS = [
    "scroll_n",
    "scroll_dur_mean",    "scroll_dur_std",
    "scroll_traj_mean",   "scroll_traj_std",
    "scroll_sdist_mean",  "scroll_sdist_std",
    "scroll_vmean_mean",  "scroll_vmean_std",
    "scroll_vmax_mean",   "scroll_vmax_std",
    "scroll_vlast5_mean", "scroll_vlast5_std",
    "scroll_mrl_mean",    "scroll_mrl_std",
    "scroll_afirst5_mean","scroll_afirst5_std",
    "scroll_dir_circmean","scroll_dir_circstd",
    "scroll_frac_up", "scroll_frac_down", "scroll_frac_left", "scroll_frac_right",
]
KEY_COLS = [
    "key_n",
    "key_inter_mean", "key_inter_std", "key_inter_median", "key_inter_p25", "key_inter_p75",
    "key_delete_rate", "key_typing_speed", "key_burst_rate",
]

FEATURE_COLS = TAP_COLS + SCROLL_COLS + KEY_COLS
FEAT_DIM     = len(FEATURE_COLS)

assert FEAT_DIM == 48
assert len(TAP_COLS)    == 16
assert len(SCROLL_COLS) == 23
assert len(KEY_COLS)    == 9

_csv_cache: dict[Path, pd.DataFrame | None] = {}

def _load_csv(user_dir: Path) -> pd.DataFrame | None:
    if user_dir in _csv_cache:
        return _csv_cache[user_dir]
    p = user_dir / "touch_session_features.csv"
    if not p.exists():
        _csv_cache[user_dir] = None
        return None
    try:
        df = pd.read_csv(p)
    except Exception as e:
        print(f"  Warning: {p}: {e}")
        _csv_cache[user_dir] = None
        return None
    if len(df) == 0 or "session_id" not in df.columns:
        _csv_cache[user_dir] = None
        return None
    for c in FEATURE_COLS:
        if c not in df.columns:
            df[c] = 0.0
    _csv_cache[user_dir] = df
    return df

def clear_cache():
    _csv_cache.clear()

def strip_user_prefix(prefixed: str, user_id: str) -> str:
    pre = f"{user_id}_"
    return prefixed[len(pre):] if prefixed.startswith(pre) else prefixed

def build_session_features(user_dir: Path,
                            session_ids) -> np.ndarray | None:
    """
    Tra ve vector 48-D cho tap session_ids.
    None neu khong co du lieu touch.
    """
    df = _load_csv(user_dir)
    if df is None:
        return None
    targets = {str(s) for s in session_ids}
    matched = df[df["session_id"].astype(str).isin(targets)]
    if len(matched) == 0:
        return None
    mat = matched[FEATURE_COLS].to_numpy(dtype=np.float64)
    vec = mat[0] if len(mat) == 1 else mat.mean(axis=0)
    if np.any(np.isnan(vec)):
        vec = np.nan_to_num(vec, nan=0.0)
    return vec

In [ ]:
%%writefile fusion.py
"""
fusion.py — Score-level fusion giữa inertial và touch.

fuse(s_inertial, s_touch, w) = w * s_inertial + (1 - w) * s_touch

find_best_w() grid-search w trong [0, 1] để cực đại AUC trên val set.
Khi tie xảy ra (vd một modal đã AUC=1.0 → mọi w cho cùng AUC), ưu tiên
w gần 0.5 nhất để fusion không thoái hóa về single modality.
"""
import numpy as np
from sklearn.metrics import roc_auc_score
from config import FUSION_STEPS, FUSION_TIE_TOLERANCE

def fuse(s_inertial: np.ndarray,
         s_touch: np.ndarray,
         w: float) -> np.ndarray:
    """Linear fusion: w * inertial + (1-w) * touch."""
    return (w * s_inertial + (1.0 - w) * s_touch).astype(np.float32)

def search_weight(s_inertial_val: np.ndarray,
                  s_touch_val: np.ndarray,
                  y_val: np.ndarray,
                  steps: int = FUSION_STEPS) -> tuple:
    """
    Grid search w ∈ [0, 1] tối ưu AUC trên val set.
    Tie-break: ưu tiên w gần 0.5 (fusion cân bằng) thay vì lấy w đầu tiên.

    Returns
    -------
    (best_w, best_val_auc)
    """
    best_w    = 0.5
    best_auc  = -1.0
    best_dist = 1.0

    for w in np.linspace(0.0, 1.0, steps):
        s = fuse(s_inertial_val, s_touch_val, w)
        try:
            auc = roc_auc_score(y_val, s)
        except ValueError:
            continue

        dist = abs(w - 0.5)

        if auc > best_auc + FUSION_TIE_TOLERANCE:

            best_auc, best_w, best_dist = auc, float(w), dist
        elif abs(auc - best_auc) <= FUSION_TIE_TOLERANCE and dist < best_dist:

            best_w, best_dist = float(w), dist

    return best_w, float(best_auc)

In [ ]:
%%writefile metrics.py
"""
metrics.py — FAR, FRR, EER, AUC.

compute_auc(y, scores)        — wrapper roc_auc_score, fallback 0.5 nếu lỗi
compute_eer(y, scores)        — interpolate đường ROC để tìm điểm FAR = FRR
compute_far_frr(y, scores, t) — FAR và FRR tại threshold t cụ thể

Khi không tìm được EER threshold hợp lệ (vd y chỉ có 1 class), fallback
về median(scores) thay vì 0.5 cứng — giữ FAR/FRR sát phân bố thực.
"""

import numpy as np
from sklearn.metrics import roc_auc_score, roc_curve

def compute_auc(y_true: np.ndarray, scores: np.ndarray) -> float:
    try:
        return float(roc_auc_score(y_true, scores))
    except ValueError:
        return 0.5

def _clean_roc(y_true: np.ndarray, scores: np.ndarray):
    """Lấy roc_curve và bỏ điểm có threshold = inf."""
    fpr, tpr, thresholds = roc_curve(y_true, scores, pos_label=1)
    mask = np.isfinite(thresholds)
    return fpr[mask], tpr[mask], thresholds[mask]

def compute_eer(y_true: np.ndarray, scores: np.ndarray) -> float:
    """EER bằng nội suy tuyến tính tại giao điểm FAR = FRR."""
    fpr, tpr, thresholds = _clean_roc(y_true, scores)
    if len(fpr) < 2:
        return 0.5
    fnr   = 1.0 - tpr
    diffs = fpr - fnr
    for i in range(len(diffs) - 1):
        if diffs[i] * diffs[i + 1] <= 0:
            d0, d1 = diffs[i], diffs[i + 1]
            if d0 == d1:
                eer = (fpr[i] + fnr[i]) / 2
            else:
                t = d0 / (d0 - d1)
                eer = fpr[i] + t * (fpr[i + 1] - fpr[i])
            return float(eer)
    idx = np.argmin(np.abs(diffs))
    return float((fpr[idx] + fnr[idx]) / 2)

def compute_far_frr_at_threshold(y_true: np.ndarray,
                                   scores: np.ndarray,
                                   threshold: float) -> tuple:
    preds = (scores >= threshold).astype(int)
    tp = int(((preds == 1) & (y_true == 1)).sum())
    fp = int(((preds == 1) & (y_true == 0)).sum())
    fn = int(((preds == 0) & (y_true == 1)).sum())
    tn = int(((preds == 0) & (y_true == 0)).sum())
    far = fp / (fp + tn + 1e-10)
    frr = fn / (fn + tp + 1e-10)
    return float(far), float(frr)

def find_eer_threshold(y_true: np.ndarray, scores: np.ndarray) -> float:
    """Threshold tại EER. Bỏ inf trước khi interpolate."""
    fpr, tpr, thresholds = _clean_roc(y_true, scores)

    if len(thresholds) < 2:
        return float(np.median(scores)) if len(scores) > 0 else 0.5

    fnr   = 1.0 - tpr
    diffs = fpr - fnr
    for i in range(len(diffs) - 1):
        if diffs[i] * diffs[i + 1] <= 0:
            d0, d1 = diffs[i], diffs[i + 1]
            t = 0.5 if d0 == d1 else d0 / (d0 - d1)
            val = thresholds[i] + t * (thresholds[i + 1] - thresholds[i])
            if np.isfinite(val):
                return float(val)
            return float(thresholds[i])

    idx = np.argmin(np.abs(diffs))
    res = float(thresholds[idx])
    return res if np.isfinite(res) else float(np.median(scores))

In [ ]:
%%writefile models.py
"""
models.py — Backbone architectures cho inertial encoder.

BackboneCNN      : Conv1D × 3 + AdaptiveAvgPool   (baseline)
BackboneConvLSTM : Conv1D × 2 + LSTM              (temporal cải tiến)

Cả hai có interface giống nhau:
  Input  : (batch, 9 channels, WINDOW_SIZE timesteps)
  Output : (batch, 128) embedding

Adaptive pooling / LSTM cho phép hoạt động với mọi độ dài sequence.
"""
import torch
import torch.nn as nn

from config import EMBED_DIM

N_CHANNELS = 9

class BackboneCNN(nn.Module):
    """
    Input (batch, 9, T)  -- T = WINDOW_SIZE (200 voi pipeline v4 walking)
      Conv1d(9->64, k=5)    + BN + ReLU + MaxPool(2)         -> (batch, 64, T/2)
      Conv1d(64->128, k=3)  + BN + ReLU + MaxPool(2)         -> (batch,128, T/4)
      Conv1d(128->128, k=3) + BN + ReLU + AdaptiveAvgPool(1) -> (batch,128)
      Dropout + Linear(128 -> n_users)                        -> logits
    """

    def __init__(self, n_users: int, n_channels: int = N_CHANNELS,
                 embed_dim: int = EMBED_DIM, dropout: float = 0.4):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv1d(n_channels, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=2),

            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=2),

            nn.Conv1d(128, embed_dim, kernel_size=3, padding=1),
            nn.BatchNorm1d(embed_dim),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
        )
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(embed_dim, n_users)

    def forward(self, x):
        emb = self.encoder(x)
        logits = self.classifier(self.dropout(emb))
        return logits, emb

    def embed(self, x: torch.Tensor) -> torch.Tensor:
        """Return 128-D embedding only. Used by backbone_train.extract_embeddings()."""
        return self.encoder(x)

    def count_params(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

class _ConvLSTMEncoder(nn.Module):
    """Conv x 2 giam sequence T->T/4, LSTM hoc temporal dependencies dai han.

    Flow: (B,9,T) -> conv -> (B,128,T/4) -> transpose -> (B,T/4,128)
          -> LSTM -> last hidden -> (B,embed_dim)
    """

    def __init__(self, n_channels: int, embed_dim: int,
                 dropout: float, bidirectional: bool):
        super().__init__()

        lstm_hidden = embed_dim // 2 if bidirectional else embed_dim

        self.conv = nn.Sequential(
            nn.Conv1d(n_channels, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(2),

            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(2),
        )
        self.lstm = nn.LSTM(
            input_size=128,
            hidden_size=lstm_hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=bidirectional,
        )
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv(x)
        x = x.permute(0, 2, 1)
        _, (h, _) = self.lstm(x)
        if h.shape[0] > 1:
            h = torch.cat([h[0], h[1]], dim=-1)
        else:
            h = h.squeeze(0)
        return self.drop(h)

class BackboneConvLSTM(nn.Module):
    """Conv1D x 2 + LSTM -- hoc tot hon pattern tuan tu dai han (gait cycle).

    Cung interface voi BackboneCNN: forward() tra (logits, embedding).
    """

    def __init__(self, n_users: int, n_channels: int = N_CHANNELS,
                 embed_dim: int = EMBED_DIM, dropout: float = 0.4,
                 bidirectional: bool = False):
        super().__init__()
        self.encoder = _ConvLSTMEncoder(n_channels, embed_dim, dropout, bidirectional)
        self.classifier = nn.Linear(embed_dim, n_users)

    def forward(self, x):
        emb = self.encoder(x)
        logits = self.classifier(emb)
        return logits, emb

    def embed(self, x: torch.Tensor) -> torch.Tensor:
        """Return 128-D embedding only. Used by backbone_train.extract_embeddings()."""
        return self.encoder(x)

    def count_params(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

BACKBONE_REGISTRY = {
    'cnn':         BackboneCNN,
    'convlstm':    BackboneConvLSTM,
    'convlstm_bi': lambda **kw: BackboneConvLSTM(bidirectional=True, **kw),
}

def build_backbone(arch: str, n_users: int, **kwargs) -> nn.Module:
    """Khoi tao backbone theo ten: 'cnn' | 'convlstm' | 'convlstm_bi'."""
    if arch not in BACKBONE_REGISTRY:
        raise ValueError(f"arch phai la {list(BACKBONE_REGISTRY)}, nhan: {arch!r}")
    return BACKBONE_REGISTRY[arch](n_users=n_users, **kwargs)

In [ ]:
%%writefile backbone_train.py
"""
backbone_train.py — Stage 1: train multi-class user-identification backbone

Mục đích: dạy CNN 1D học general "behavioral features" của các user. Sau
khi train xong, vứt classifier head, giữ lại encoder (output 128-D
embedding) — đây là phần được ship với app.

Train task: phân loại user-id (n_users lớp) trên TRAIN sessions của TẤT CẢ user.
            VAL set = VAL sessions (dùng để early stop).

Không nhúng nhãn activity (sitting/standing/walking) vào loss — backbone
tự học activity-agnostic representation qua việc nhìn data hỗn hợp.

Output:
  • model    : BackboneCNN đã train
  • val_acc  : multi-class accuracy trên val (chỉ để debug, không phải
               metric chính)
"""

import numpy as np
import torch
import torch.nn as nn

from models   import build_backbone
from dataset  import prepare_tensors
from config   import (
    BACKBONE_ARCH, BACKBONE_EPOCHS, BACKBONE_LR, BACKBONE_BATCH, BACKBONE_PATIENCE,
    EMBED_DIM, PRELOAD_TO_DEVICE, USE_AMP, ENABLE_TF32, USE_FUSED_ADAM,
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if DEVICE.type == "cuda" and ENABLE_TF32:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True

def _make_grad_scaler():
    """torch.amp.GradScaler('cuda') tu PyTorch 2.4 — fallback len torch.cuda.amp."""
    try:
        return torch.amp.GradScaler("cuda")
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler()

def _autocast_ctx(dtype=torch.float16):
    """torch.amp.autocast('cuda', dtype=...) tu PyTorch 2.4 — fallback len torch.cuda.amp."""
    try:
        return torch.amp.autocast("cuda", dtype=dtype)
    except (AttributeError, TypeError):
        return torch.cuda.amp.autocast(dtype=dtype)

def _build_multiclass_dataset(users: dict,
                              user_to_id: dict[str, int],
                              session_filter: set[str]) -> tuple[np.ndarray, np.ndarray]:
    """
    Gộp data của TẤT CẢ user, chỉ lấy windows thuộc session_filter.
    Trả về (X, y) với y = user-id (int) theo user_to_id mapping.
    """
    X_list, y_list = [], []
    for uid, data in users.items():
        mask = np.isin(data["session"].astype(str), list(session_filter))
        if not mask.any():
            continue
        X_list.append(data["X"][mask])
        y_list.append(np.full(int(mask.sum()), user_to_id[uid], dtype=np.int64))

    if not X_list:
        raise ValueError("Không có window nào sau session_filter — pipeline lỗi.")

    X = np.concatenate(X_list).astype(np.float32)
    y = np.concatenate(y_list)
    return X, y

def train_backbone(users: dict,
                   user_to_id: dict[str, int],
                   train_sessions_global: set[str],
                   val_sessions_global: set[str],
                   seed: int = 42,
                   verbose: bool = True) -> tuple[nn.Module, float]:
    """
    Train backbone CNN multi-class identification.

    Parameters
    ----------
    users                : dict trả về từ load_users()
    user_to_id           : {user_id_str: int_label}
    train_sessions_global: tập sessions (PREFIXED) dùng để train
    val_sessions_global  : tập sessions (PREFIXED) dùng để early-stop
    seed                 : random seed

    Returns
    -------
    (model, best_val_acc)
    """
    torch.manual_seed(seed)
    np.random.seed(seed)

    n_users = len(user_to_id)

    X_tr, y_tr = _build_multiclass_dataset(users, user_to_id, train_sessions_global)
    X_val, y_val = _build_multiclass_dataset(users, user_to_id, val_sessions_global)

    if verbose:
        print(f"    Backbone train: {len(X_tr):,} windows  "
              f"val: {len(X_val):,} windows  n_users={n_users}")

    model = build_backbone(BACKBONE_ARCH, n_users=n_users, embed_dim=EMBED_DIM).to(DEVICE)
    if verbose:
        print(f"    Backbone params: {model.count_params():,}")

    criterion = nn.CrossEntropyLoss()

    optimizer_kwargs = dict(lr=BACKBONE_LR, weight_decay=1e-4)
    if USE_FUSED_ADAM and DEVICE.type == "cuda":
        try:
            optimizer = torch.optim.Adam(model.parameters(), fused=True, **optimizer_kwargs)
        except (TypeError, RuntimeError):
            optimizer = torch.optim.Adam(model.parameters(), **optimizer_kwargs)
    else:
        optimizer = torch.optim.Adam(model.parameters(), **optimizer_kwargs)

    use_amp = USE_AMP and DEVICE.type == "cuda"
    scaler  = _make_grad_scaler() if use_amp else None

    X_tr_t  = torch.from_numpy(np.ascontiguousarray(X_tr.transpose(0, 2, 1))).float()
    y_tr_t  = torch.from_numpy(y_tr).long()
    X_val_t = torch.from_numpy(np.ascontiguousarray(X_val.transpose(0, 2, 1))).float()
    y_val_t = torch.from_numpy(y_val).long()

    if PRELOAD_TO_DEVICE and DEVICE.type == "cuda":
        X_tr_t  = X_tr_t.to(DEVICE)
        y_tr_t  = y_tr_t.to(DEVICE)
        X_val_t = X_val_t.to(DEVICE)
        y_val_t = y_val_t.to(DEVICE)

    best_val_acc = -1.0
    best_state   = None
    no_improve   = 0

    for epoch in range(1, BACKBONE_EPOCHS + 1):

        model.train()
        n = X_tr_t.size(0)
        perm = torch.randperm(n, device=X_tr_t.device)
        total_loss = 0.0

        for i in range(0, n, BACKBONE_BATCH):
            idx = perm[i:i + BACKBONE_BATCH]
            xb = X_tr_t.index_select(0, idx)
            yb = y_tr_t.index_select(0, idx)

            optimizer.zero_grad(set_to_none=True)

            if use_amp:
                with _autocast_ctx(dtype=torch.float16):
                    logits, _ = model(xb)
                    loss = criterion(logits, yb)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                logits, _ = model(xb)
                loss = criterion(logits, yb)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            total_loss += loss.item() * yb.size(0)

        model.eval()
        with torch.no_grad():
            n_val = X_val_t.size(0)
            n_correct = 0
            for i in range(0, n_val, BACKBONE_BATCH):
                xb = X_val_t[i:i + BACKBONE_BATCH]
                yb = y_val_t[i:i + BACKBONE_BATCH]
                if use_amp:
                    with _autocast_ctx(dtype=torch.float16):
                        logits, _ = model(xb)
                else:
                    logits, _ = model(xb)
                preds = logits.argmax(dim=1)
                n_correct += int((preds == yb).sum().item())
            val_acc = n_correct / n_val

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1

        if verbose and (epoch % 5 == 0 or epoch == 1):
            print(f"      ep{epoch:3d}  loss={total_loss/n:.4f}  "
                  f"val_acc={val_acc:.4f}  best={best_val_acc:.4f}  "
                  f"patience={no_improve}/{BACKBONE_PATIENCE}")

        if no_improve >= BACKBONE_PATIENCE:
            if verbose:
                print(f"      Early stop tại epoch {epoch}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return model, best_val_acc

@torch.no_grad()
def extract_embeddings(model: nn.Module,
                       X: np.ndarray,
                       batch_size: int = 256) -> np.ndarray:
    """
    Map (N, WINDOW_SIZE, 9) → (N, 128) embedding.

    Backbone đã ở DEVICE; X được move sang DEVICE theo batch.
    """
    model.eval()
    X_t = torch.from_numpy(np.ascontiguousarray(X.transpose(0, 2, 1))).float()

    use_amp = USE_AMP and DEVICE.type == "cuda"

    embs = []
    for i in range(0, len(X_t), batch_size):
        xb = X_t[i:i + batch_size].to(DEVICE, non_blocking=False)
        if use_amp:
            with _autocast_ctx(dtype=torch.float16):
                e = model.embed(xb).float()
        else:
            e = model.embed(xb)
        embs.append(e.cpu().numpy())

    return np.concatenate(embs).astype(np.float32)

In [ ]:
%%writefile per_user_eval_cosine.py
"""Đánh giá xác thực khớp hàm quyết định trên thiết bị (cosine-to-anchors + EWMA),
hỗ trợ hai giao thức open-set và closed-set, báo cáo riêng inertial/touch/fusion."""

import numpy as np
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

from touch_features import build_session_features, strip_user_prefix, FEAT_DIM
from fusion         import search_weight, fuse
from metrics        import compute_auc, compute_eer, compute_far_frr_at_threshold, find_eer_threshold
from config         import (
    RF_N_ESTIMATORS, RF_MAX_FEATURES, RF_CLASS_WEIGHT, RF_MIN_SAMPLES_LEAF_TOUCH,
)

SIGMOID_SCALE = 8.0
SIGMOID_BIAS  = 0.30
ANCHOR_COUNT  = 6
EWMA_ALPHA    = 0.8
EWMA_WINDOW   = 5

def _l2norm(M: np.ndarray) -> np.ndarray:
    return M / (np.linalg.norm(M, axis=-1, keepdims=True) + 1e-9)

def mean_cosine_to_anchors(emb: np.ndarray, anchors: np.ndarray) -> np.ndarray:
    """emb: (N,128), anchors: (A,128) → (N,) mean cosine sim tới các anchor."""
    if len(anchors) == 0:
        return np.full(len(emb), 0.0, dtype=np.float32)
    e = _l2norm(emb.astype(np.float64))
    a = _l2norm(anchors.astype(np.float64))
    return (e @ a.T).mean(axis=1).astype(np.float32)

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def score_inertial_cosine(emb: np.ndarray, anchors: np.ndarray) -> np.ndarray:
    """Giống hệt: sigmoid(SCALE*(meanSim - BIAS))."""
    sim = mean_cosine_to_anchors(emb, anchors)
    return sigmoid(SIGMOID_SCALE * (sim - SIGMOID_BIAS)).astype(np.float32)

def ewma_last(scores_in_order: np.ndarray,
              alpha: float = EWMA_ALPHA,
              window: int = EWMA_WINDOW) -> float:
    """Giá trị aggregate ở cuối chuỗi: trọng số alpha^k cho mẫu cách cuối k bước."""
    if len(scores_in_order) == 0:
        return 0.5
    tail = scores_in_order[-window:]
    k = np.arange(len(tail) - 1, -1, -1)
    w = alpha ** k
    return float((w * tail).sum() / w.sum())

def aggregate_per_session(window_scores: np.ndarray,
                          session_ids: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """
    Gom điểm window → 1 điểm/ session bằng EWMA (theo thứ tự xuất hiện trong
    mảng — giả định mảng đã giữ thứ tự thời gian của từng session).
    Trả về (sess_scores, sess_unique).
    """
    sess_unique = []
    sess_scores = []
    for s in dict.fromkeys(session_ids.tolist()):
        mask = session_ids == s
        sess_scores.append(ewma_last(window_scores[mask]))
        sess_unique.append(s)
    return np.asarray(sess_scores, np.float32), np.asarray(sess_unique)

def evaluate_owner_cosine(owner_id: str,
                          users: dict,
                          backbone_model,
                          enroll_sessions:  dict[str, list[str]],
                          train_sessions:   dict[str, list[str]],
                          val_sessions:     dict[str, list[str]],
                          test_sessions:    dict[str, list[str]],
                          impostor_pool_touch: np.ndarray,
                          touch_scaler: StandardScaler,
                          data_dir: Path,
                          seed: int = 42,
                          verbose: bool = True) -> dict:
    """
    enroll_sessions[owner_id] : session(s) dùng để DỰNG ANCHOR (mô phỏng enroll).
                                Tách rời train/val/test → không rò rỉ.
    Nhánh inertial KHÔNG còn RF, KHÔNG cần impostor_pool_inertial.
    """
    from backbone_train import extract_embeddings

    owner = users[owner_id]
    user_dir = data_dir / owner_id

    def _win(uid, sess):
        d = users[uid]
        m = np.isin(d["session"].astype(str), list(sess))
        return d["X"][m], d["session"][m]

    enr_X, _ = _win(owner_id, enroll_sessions[owner_id])
    if len(enr_X) == 0:
        raise ValueError(f"Owner {owner_id}: enroll session rỗng")
    enr_emb = extract_embeddings(backbone_model, enr_X)
    if len(enr_emb) > ANCHOR_COUNT:
        idx = np.linspace(0, len(enr_emb) - 1, ANCHOR_COUNT).round().astype(int)
        anchors = enr_emb[idx]
    else:
        anchors = enr_emb

    rf_touch = None
    own_touch_tr, _ = _touch_vectors(user_dir, owner_id, train_sessions[owner_id])
    if len(own_touch_tr) > 0:
        own_t_scaled = touch_scaler.transform(own_touch_tr)
        X_t = np.concatenate([own_t_scaled, impostor_pool_touch])
        y_t = np.concatenate([np.ones(len(own_t_scaled), int),
                              np.zeros(len(impostor_pool_touch), int)])
        rf_touch = RandomForestClassifier(
            n_estimators=RF_N_ESTIMATORS, max_features=RF_MAX_FEATURES,
            class_weight=RF_CLASS_WEIGHT, min_samples_leaf=RF_MIN_SAMPLES_LEAF_TOUCH,
            random_state=seed, n_jobs=-1)
        rf_touch.fit(X_t, y_t)

    def _split(sess_map, name):
        Xs, ss, ys = [], [], []
        for uid, sl in sess_map.items():
            if not sl:
                continue
            X, s = _win(uid, sl)
            if len(X) == 0:
                continue
            Xs.append(X); ss.append(s)
            ys.append(np.full(len(X), 1 if uid == owner_id else 0, np.float32))
        if not Xs:
            raise ValueError(f"{owner_id}: {name} rỗng")
        return np.concatenate(Xs), np.concatenate(ss), np.concatenate(ys)

    val_X, val_s, val_y   = _split(val_sessions,  "val")
    test_X, test_s, test_y = _split(test_sessions, "test")

    s_i_val  = score_inertial_cosine(extract_embeddings(backbone_model, val_X),  anchors)
    s_i_test = score_inertial_cosine(extract_embeddings(backbone_model, test_X), anchors)

    s_t_val  = _touch_scores(rf_touch, val_s,  users, data_dir, touch_scaler)
    s_t_test = _touch_scores(rf_touch, test_s, users, data_dir, touch_scaler)

    best_w, _ = search_weight(s_i_val, s_t_val, val_y)
    s_fin = fuse(s_i_test, s_t_test, best_w)

    def _session_metrics(win_scores):
        sc, uni = aggregate_per_session(win_scores, test_s)
        lab = np.array([test_y[test_s == s][0] for s in uni], np.float32)
        thr = find_eer_threshold(lab, sc)
        far, frr = compute_far_frr_at_threshold(lab, sc, thr)
        return dict(auc=float(compute_auc(lab, sc)),
                    eer=float(compute_eer(lab, sc)),
                    far=float(far), frr=float(frr), thr=float(thr),
                    _scores=sc, _labels=lab)

    m_iner = _session_metrics(s_i_test)
    m_touch = _session_metrics(s_t_test)
    m_fuse = _session_metrics(s_fin)

    if verbose:
        print(f"  {owner_id:>10}  "
              f"INER[AUC={m_iner['auc']:.3f} EER={m_iner['eer']:.3f}]  "
              f"TOUCH[AUC={m_touch['auc']:.3f} EER={m_touch['eer']:.3f}]  "
              f"FUSE[AUC={m_fuse['auc']:.3f} EER={m_fuse['eer']:.3f} "
              f"FAR={m_fuse['far']:.3f} FRR={m_fuse['frr']:.3f}]  w={best_w:.2f}")

    return dict(
        owner_id=owner_id, fusion_w=float(best_w), has_touch=(rf_touch is not None),

        auc_inertial=m_iner['auc'], eer_inertial=m_iner['eer'],
        far_inertial=m_iner['far'], frr_inertial=m_iner['frr'],

        auc_touch=m_touch['auc'], eer_touch=m_touch['eer'],
        far_touch=m_touch['far'], frr_touch=m_touch['frr'],

        auc_session=m_fuse['auc'], eer_session=m_fuse['eer'],
        far_session=m_fuse['far'], frr_session=m_fuse['frr'],
        thr_session=m_fuse['thr'],

        _sess_scores=m_fuse['_scores'], _sess_labels=m_fuse['_labels'],
        _iner_scores=m_iner['_scores'], _touch_scores=m_touch['_scores'],
    )

import pandas as _pd

TAP_CHUNK = 20
SCR_CHUNK = 10
MIN_TAP_IN_CHUNK = 5
_SCROLL_NUM = ["duration_ms", "traj", "v_mean", "v_max", "straight", "mrl",
               "v_start", "v_last5", "peak_ratio", "a_first5", "disp_y"]
TOUCH_SUBWIN_DIM = 3 * 3 + 1 + len(_SCROLL_NUM) * 2 + 1

_raw_cache: dict = {}

def _load_raw(user_dir):
    key = str(user_dir)
    if key in _raw_cache:
        return _raw_cache[key]
    try:
        tp = _pd.read_csv(user_dir / "tap_gestures.csv")
        sc = _pd.read_csv(user_dir / "scroll_gestures.csv")
    except Exception:
        _raw_cache[key] = (None, None); return None, None
    _raw_cache[key] = (tp, sc); return tp, sc

def _chunk_feats(tp_c, sc_c):
    f = []
    hold = tp_c["hold_ms"].values if len(tp_c) else np.array([0.0])
    disp = tp_c["displacement"].values if len(tp_c) else np.array([0.0])
    iti = (np.diff(np.sort(tp_c["timestamp_ms"].values))
           if len(tp_c) > 1 else np.array([0.0]))
    for arr in (hold, disp, iti):
        a = arr if len(arr) else np.array([0.0])
        f += [float(np.mean(a)), float(np.std(a)), float(np.median(a))]
    f += [float(len(tp_c))]
    for col in _SCROLL_NUM:
        a = sc_c[col].values if (len(sc_c) and col in sc_c.columns) else np.array([0.0])
        a = a if len(a) else np.array([0.0])
        f += [float(np.mean(a)), float(np.std(a))]
    f += [float(len(sc_c))]
    return np.asarray(f, np.float64)

def touch_subwindows(user_dir, sess_unprefixed):
    """Trả về (n_subwindow, TOUCH_SUBWIN_DIM) cho các session chỉ định."""
    tp, sc = _load_raw(user_dir)
    if tp is None:
        return np.zeros((0, TOUCH_SUBWIN_DIM))
    out = []
    for s in sess_unprefixed:
        t = tp[tp["session_id"] == s].reset_index(drop=True)
        k = sc[sc["session_id"] == s].reset_index(drop=True) if "session_id" in sc.columns else sc.iloc[0:0]
        if len(t) == 0:
            continue
        n = max(1, len(t) // TAP_CHUNK)
        for i in range(n):
            tc = t.iloc[i * TAP_CHUNK:(i + 1) * TAP_CHUNK]
            kc = k.iloc[i * SCR_CHUNK:(i + 1) * SCR_CHUNK] if len(k) else k
            if len(tc) >= MIN_TAP_IN_CHUNK:
                out.append(_chunk_feats(tc, kc))
    return np.asarray(out, np.float64) if out else np.zeros((0, TOUCH_SUBWIN_DIM))

def _touch_vectors(user_dir, uid, sess_pref):
    """Owner positives = TẤT CẢ sub-window của các session enroll."""
    unpref = {strip_user_prefix(s, uid) for s in sess_pref}
    M = touch_subwindows(user_dir, unpref)
    return M, None

def _touch_scores(rf_touch, sess_ids, users, data_dir, scaler):
    """Điểm touch mỗi session = trung bình prob RF trên các sub-window của session."""
    if rf_touch is None:
        return np.full(len(sess_ids), 0.5, np.float32)
    s2u = {s: uid for uid, d in users.items() for s in d["session"].astype(str)}
    cache = {}
    def one(s):
        if s in cache: return cache[s]
        uid = s2u.get(s)
        if uid is None:
            cache[s] = 0.5; return 0.5
        M = touch_subwindows(data_dir / uid, {strip_user_prefix(s, uid)})
        if len(M) == 0:
            cache[s] = 0.5; return 0.5
        probs = rf_touch.predict_proba(scaler.transform(M))[:, 1]
        sc = float(np.mean(probs))
        cache[s] = sc; return sc
    return np.array([one(str(s)) for s in sess_ids], np.float32)

def _run_eval_on_population(users_pop: dict,
                            backbone_model,
                            impostor_pool_touch: np.ndarray,
                            touch_scaler: StandardScaler,
                            data_dir: Path,
                            enroll_frac: float,
                            val_frac: float,
                            test_frac: float,
                            seed: int,
                            verbose: bool) -> dict:
    """Lõi chung: mỗi owner trong users_pop được enroll rồi test; impostor lúc
    test = các owner KHÁC trong cùng users_pop. Giao thức session-disjoint.
    - OPEN-SET  : users_pop = danh tính HELD-OUT (backbone chưa thấy).
    - CLOSED-SET: users_pop = danh tính SEEN (backbone đã train trên họ).
    """
    rng = np.random.default_rng(seed)
    hid = list(users_pop.keys())

    def split_sessions(uid):
        ss = np.unique(users_pop[uid]["session"]).astype(str)
        rng.shuffle(ss)
        n = len(ss)
        n_en = max(1, int(round(n * enroll_frac)))
        en, rest = ss[:n_en], ss[n_en:]
        n_te = max(1, int(round(len(rest) * (test_frac / (test_frac + val_frac + 1e-9))))) if (val_frac > 0) else len(rest)
        te = rest[:n_te]; va = rest[n_te:]
        return list(en), list(va), list(te)

    en_map, va_map, te_map = {}, {}, {}
    for uid in hid:
        en_map[uid], va_map[uid], te_map[uid] = split_sessions(uid)

    results = []
    for owner in hid:
        if len(te_map[owner]) == 0:
            continue
        test_sessions = {owner: te_map[owner]}
        for other in hid:
            if other != owner and te_map[other]:
                test_sessions[other] = te_map[other]
        val_sessions = {owner: (va_map[owner] or te_map[owner])}
        for other in hid:
            if other != owner:
                val_sessions[other] = (va_map[other] or te_map[other])
        try:
            r = evaluate_owner_cosine(
                owner, users_pop, backbone_model,
                enroll_sessions={owner: en_map[owner]},
                train_sessions={owner: en_map[owner]},
                val_sessions=val_sessions, test_sessions=test_sessions,
                impostor_pool_touch=impostor_pool_touch,
                touch_scaler=touch_scaler, data_dir=data_dir, seed=seed,
                verbose=verbose)
            results.append(r)
        except ValueError as e:
            if verbose:
                print(f"  [skip] {owner}: {e}")

    return summarize_and_calibrate(results, verbose=verbose)

def run_open_set_eval(users_heldout: dict,
                      backbone_model,
                      impostor_pool_touch: np.ndarray,
                      touch_scaler: StandardScaler,
                      data_dir: Path,
                      enroll_frac: float = 0.34,
                      val_frac: float = 0.16,
                      test_frac: float = 0.50,
                      seed: int = 42,
                      verbose: bool = True) -> dict:
    """OPEN-SET: users_heldout là danh tính GIỮ LẠI — backbone & impostor pool
    CHƯA TỪNG thấy. Owner và 'kẻ tấn công' lúc test đều là người lạ hoàn toàn.
    → Kịch bản khó nhất, sát với app ship sẵn model chưa có owner."""
    return _run_eval_on_population(
        users_heldout, backbone_model, impostor_pool_touch, touch_scaler,
        data_dir, enroll_frac, val_frac, test_frac, seed, verbose)

def run_closed_set_eval(users_seen: dict,
                        backbone_model,
                        impostor_pool_touch: np.ndarray,
                        touch_scaler: StandardScaler,
                        data_dir: Path,
                        enroll_frac: float = 0.34,
                        val_frac: float = 0.16,
                        test_frac: float = 0.50,
                        seed: int = 42,
                        verbose: bool = True) -> dict:
    """CLOSED-SET: users_seen là danh tính ĐÃ train backbone. Owner enroll từ
    vài session, test trên session khác (session-disjoint, KHÔNG rò rỉ cửa sổ),
    impostor = các user SEEN khác. Đây là giao thức các nghiên cứu tham khảo
    thường đo → con số so sánh được với mục tiêu < 5%.
    Lưu ý: dễ hơn open-set vì backbone đã học cách tách chính những người này."""
    return _run_eval_on_population(
        users_seen, backbone_model, impostor_pool_touch, touch_scaler,
        data_dir, enroll_frac, val_frac, test_frac, seed, verbose)

def summarize_and_calibrate(results: list[dict], verbose: bool = True) -> dict:
    """Gộp điểm session toàn bộ owner → metric 3 nhánh + ngưỡng vận hành chung."""
    if not results:
        return {"n_owners": 0}
    all_s = np.concatenate([r["_sess_scores"] for r in results])
    all_y = np.concatenate([r["_sess_labels"] for r in results])
    all_i = np.concatenate([r["_iner_scores"] for r in results])
    all_t = np.concatenate([r["_touch_scores"] for r in results])

    def _global(scores):
        thr = find_eer_threshold(all_y, scores)
        far, frr = compute_far_frr_at_threshold(all_y, scores, thr)
        return dict(auc=float(compute_auc(all_y, scores)),
                    eer=float(compute_eer(all_y, scores)),
                    far=float(far), frr=float(frr), thr=float(thr))

    g_iner = _global(all_i)
    g_touch = _global(all_t)
    g_fuse = _global(all_s)

    thr_eer = g_fuse["thr"]
    trusted = float(np.clip(thr_eer + 0.15, 0, 1))
    warning = float(np.clip(thr_eer - 0.05, 0, 1))

    summary = dict(
        n_owners=len(results),

        inertial=g_iner,
        touch=g_touch,
        fusion=g_fuse,

        auc_global=g_fuse["auc"], eer_global=g_fuse["eer"],
        far_at_eer=g_fuse["far"], frr_at_eer=g_fuse["frr"],
        decision_threshold_eer=thr_eer,

        per_owner=[dict(owner_id=r["owner_id"],
                        eer_inertial=r["eer_inertial"], auc_inertial=r["auc_inertial"],
                        eer_touch=r["eer_touch"],       auc_touch=r["auc_touch"],
                        eer_fusion=r["eer_session"],    auc_fusion=r["auc_session"],
                        far_fusion=r["far_session"],    frr_fusion=r["frr_session"],
                        fusion_w=r["fusion_w"], has_touch=r["has_touch"]) for r in results],
        scores_genuine=all_s[all_y == 1].tolist(),
        scores_impostor=all_s[all_y == 0].tolist(),
        manifest_patch=dict(
            sigmoid_scale=SIGMOID_SCALE,
            sigmoid_bias=SIGMOID_BIAS,
            decision_threshold_eer=round(float(thr_eer), 4),
            trusted_threshold=round(trusted, 4),
            warning_threshold=round(warning, 4),
            score_aggregator_alpha=EWMA_ALPHA,
            score_aggregator_window=EWMA_WINDOW,
            anchor_count=ANCHOR_COUNT,
            score_method="cosine_similarity_to_owner_anchors",
            eval_protocol="open_set_session_disjoint_cosine_ewma",
        ),
    )
    if verbose:
        print("\n=== OPEN-SET (sau EWMA, mức session) ===")
        print(f"  owners={summary['n_owners']}")
        print(f"  INERTIAL : AUC={g_iner['auc']:.4f} EER={g_iner['eer']:.4f} "
              f"FAR={g_iner['far']:.4f} FRR={g_iner['frr']:.4f}")
        print(f"  TOUCH    : AUC={g_touch['auc']:.4f} EER={g_touch['eer']:.4f} "
              f"FAR={g_touch['far']:.4f} FRR={g_touch['frr']:.4f}")
        print(f"  FUSION   : AUC={g_fuse['auc']:.4f} EER={g_fuse['eer']:.4f} "
              f"FAR={g_fuse['far']:.4f} FRR={g_fuse['frr']:.4f}  (thr={thr_eer:.4f})")
    return summary

In [ ]:
%%writefile main.py
"""Pipeline sản phẩm: train backbone trên SEEN → export artifacts →
đánh giá OPEN-SET (người mới) và CLOSED-SET (người đã biết)."""

import argparse
import os
import sys

def _peek_context_mode() -> str:
    _p = argparse.ArgumentParser(add_help=False)
    _p.add_argument("--context", choices=["walking", "all"], default="all")
    _args, _ = _p.parse_known_args()
    return _args.context

os.environ["CONTEXT_MODE"] = _peek_context_mode()

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

from config import (
    CONTEXT_MODE, MODELS_DIR, RESULTS_DIR, EXPORT_DIR,
    TEST_SIZE, VAL_SIZE, EMBED_DIM,
    POOL_SIZE_INERTIAL, POOL_SIZE_TOUCH, SEED,
)
from dataset import load_users
from backbone_train import train_backbone, extract_embeddings
from touch_features import strip_user_prefix
from per_user_eval_cosine import run_open_set_eval, run_closed_set_eval

import torch

EXCLUDE_USERS = {"user17",
                 "user11"}

def _header(text, width=72):
    print("\n" + "═" * width + f"\n  {text}\n" + "═" * width)

def _section(text):
    print("\n" + "─" * 60 + f"\n  {text}\n" + "─" * 60)

def _split_sessions(users, seed):
    """Chia sessions của TỪNG user thành train/val/test (session-disjoint)."""
    train_sess, val_sess, test_sess = {}, {}, {}
    for uid, data in users.items():
        sessions = sorted(np.unique(data["session"]))
        if len(sessions) < 3:
            train_sess[uid], val_sess[uid], test_sess[uid] = list(sessions), [], []
            continue
        n_te = max(1, int(len(sessions) * TEST_SIZE))
        n_val = max(1, int((len(sessions) - n_te) * VAL_SIZE))
        n_tr = len(sessions) - n_te - n_val
        train_sess[uid] = list(sessions[:n_tr])
        val_sess[uid] = list(sessions[n_tr:n_tr + n_val])
        test_sess[uid] = list(sessions[n_tr + n_val:])
    return train_sess, val_sess, test_sess

def _build_pool_inertial(users, backbone, train_sessions, pool_size, seed):
    rng = np.random.default_rng(seed)
    X_list = []
    for uid, sess_list in train_sessions.items():
        if not sess_list:
            continue
        data = users[uid]
        mask = np.isin(data["session"].astype(str), list(sess_list))
        if mask.any():
            X_list.append(data["X"][mask])
    if not X_list:
        return np.zeros((0, EMBED_DIM), np.float32)
    X_all = np.concatenate(X_list)
    if len(X_all) > pool_size:
        X_all = X_all[rng.choice(len(X_all), pool_size, replace=False)]
    return extract_embeddings(backbone, X_all)

def _build_pool_touch(users, train_sessions, pool_size, data_dir, seed):
    from per_user_eval_cosine import touch_subwindows, strip_user_prefix as _strip
    rng = np.random.default_rng(seed)
    vectors = []
    for uid, sess_list in train_sessions.items():
        user_dir = data_dir / uid
        if not user_dir.exists():
            continue
        unpref = {_strip(s, uid) for s in sess_list}
        M = touch_subwindows(user_dir, unpref)
        if len(M):
            vectors.append(M)
    if not vectors:
        from per_user_eval_cosine import TOUCH_SUBWIN_DIM
        return np.zeros((0, TOUCH_SUBWIN_DIM), np.float64)
    arr = np.concatenate(vectors, axis=0)
    if len(arr) > pool_size:
        arr = arr[rng.choice(len(arr), pool_size, replace=False)]
    return arr

def _export_artifacts(backbone, pool_inertial, pool_touch, touch_scaler, seen_ids):
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    MODELS_DIR.mkdir(parents=True, exist_ok=True)
    torch.save(backbone.state_dict(), MODELS_DIR / "backbone.pt")
    np.save(EXPORT_DIR / "impostor_pool_inertial.npy", pool_inertial.astype(np.float32))
    np.save(EXPORT_DIR / "impostor_pool_touch.npy", pool_touch.astype(np.float32))
    with open(EXPORT_DIR / "touch_scaler.json", "w") as f:
        json.dump({"mean": touch_scaler.mean_.tolist(),
                   "scale": touch_scaler.scale_.tolist(),
                   "n_features": int(touch_scaler.n_features_in_)}, f, indent=2)
    with open(EXPORT_DIR / "backbone_metadata.json", "w") as f:
        json.dump({"context_mode": CONTEXT_MODE,
                   "embed_dim": EMBED_DIM,
                   "users_trained": seen_ids,
                   "touch_feat_dim": int(pool_touch.shape[1]) if len(pool_touch) else 0,
                   "framework": "pytorch",
                   "note": "Deploy: backbone.pt → ONNX → TFLite"}, f, indent=2)
    print(f"    → {EXPORT_DIR}/ (pools, scaler, metadata) + {MODELS_DIR}/backbone.pt")

def main(data_dir, held_out_ids, n_eval_runs, verbose):
    data_path = Path(data_dir)
    for d in (MODELS_DIR, RESULTS_DIR, EXPORT_DIR):
        d.mkdir(parents=True, exist_ok=True)

    _header(f"PIPELINE — context={CONTEXT_MODE!r}", 72)

    _section("Stage 1 — Load & tách SEEN / HELD-OUT")
    users_all = load_users(data_path, exclude_users=EXCLUDE_USERS,
                           context_mode=CONTEXT_MODE)
    if not users_all:
        print(f"  Không tìm thấy user trong {data_path}"); sys.exit(1)
    ids = sorted(users_all.keys())

    held = sorted(set(held_out_ids) & set(ids))
    seen = sorted(set(ids) - set(held))
    missing = sorted(set(held_out_ids) - set(ids))
    if missing:
        print(f"  ⚠ held-out không có trong data (bỏ qua): {missing}")
    if len(seen) < 2 or len(held) < 1:
        print(f"  Cần ≥2 SEEN và ≥1 HELD-OUT. SEEN={len(seen)} HELD-OUT={len(held)}")
        sys.exit(1)
    print(f"  SEEN ({len(seen)}, train backbone): {seen}")
    print(f"  HELD-OUT ({len(held)}, đánh giá người mới): {held}")

    users_seen = {u: users_all[u] for u in seen}
    users_held = {u: users_all[u] for u in held}
    seen_to_id = {u: i for i, u in enumerate(seen)}

    _section("Stage 2 — Train backbone CNN (SEEN)")
    tr, va, _ = _split_sessions(users_seen, SEED)
    tr_all = set(s for L in tr.values() for s in L)
    va_all = set(s for L in va.values() for s in L)
    t0 = time.time()
    backbone, val_acc = train_backbone(
        users_seen, seen_to_id,
        train_sessions_global=tr_all, val_sessions_global=va_all,
        seed=SEED, verbose=verbose)
    print(f"    backbone val_acc(user-id)={val_acc:.4f}  ({time.time()-t0:.1f}s)")

    _section("Stage 3 — Impostor pool + touch scaler (SEEN)")
    pool_i = _build_pool_inertial(users_seen, backbone, tr, POOL_SIZE_INERTIAL, SEED)
    pool_t_raw = _build_pool_touch(users_seen, tr, max(POOL_SIZE_TOUCH, 400), data_path, SEED)
    touch_scaler = StandardScaler()
    pool_t = touch_scaler.fit_transform(pool_t_raw) if len(pool_t_raw) else pool_t_raw
    print(f"    pool inertial={len(pool_i)}  pool touch={len(pool_t)}")

    _section("Stage 4 — Export artifacts deploy")
    _export_artifacts(backbone, pool_i, pool_t, touch_scaler, seen)

    branches = ["inertial", "touch", "fusion"]
    metrics = ["auc", "eer", "far", "frr"]

    def _eval_multi(eval_fn, users_pop, tag, prefix, extra):
        runs, first = [], None
        for k in range(n_eval_runs):
            s = eval_fn(users_pop, backbone_model=backbone,
                        impostor_pool_touch=pool_t, touch_scaler=touch_scaler,
                        data_dir=data_path, seed=SEED + k, verbose=False)
            runs.append(s)
            if k == 0:
                first = s
            print(f"    [{tag}] run {k+1}/{n_eval_runs}: "
                  f"INER EER={s['inertial']['eer']:.3f} | "
                  f"TOUCH EER={s['touch']['eer']:.3f} | "
                  f"FUSE EER={s['fusion']['eer']:.3f} AUC={s['fusion']['auc']:.3f}")

        def ms(b, m):
            v = np.array([r[b][m] for r in runs], float)
            return float(v.mean()), float(v.std())
        agg = {b: {m: ms(b, m) for m in metrics} for b in branches}

        print(f"\n  === KẾT QUẢ {tag} (mean ± std qua {n_eval_runs} seed, mức session) ===")
        print(f"    {'nhánh':<10} {'AUC':>14} {'EER':>14} {'FAR':>14} {'FRR':>14}")
        for b in branches:
            print(f"    {b:<10} " + "  ".join(
                f"{agg[b][m][0]:.3f}±{agg[b][m][1]:.3f}" for m in metrics))

        pd.DataFrame(first["per_owner"]).to_csv(
            RESULTS_DIR / f"{prefix}_per_owner.csv", index=False)
        out = dict(context_mode=CONTEXT_MODE, n_eval_runs=n_eval_runs,
                   protocol=tag,
                   summary={b: {m: {"mean": agg[b][m][0], "std": agg[b][m][1]}
                                for m in metrics} for b in branches},
                   scores_genuine=first["scores_genuine"],
                   scores_impostor=first["scores_impostor"],
                   **extra)
        if prefix == "openset":
            out["manifest_patch"] = first["manifest_patch"]
        with open(RESULTS_DIR / f"{prefix}_summary.json", "w", encoding="utf-8") as f:
            json.dump(out, f, ensure_ascii=False, indent=2)
        print(f"  Đã ghi: {RESULTS_DIR}/{prefix}_summary.json + {prefix}_per_owner.csv")
        return first

    _section(f"Stage 5 — Đánh giá OPEN-SET (người mới hoàn toàn · {n_eval_runs} seed)")
    _eval_multi(run_open_set_eval, users_held, "OPEN-SET", "openset",
                extra=dict(seen_users=seen, held_out_users=held))

    _section(f"Stage 6 — Đánh giá CLOSED-SET (người đã biết · {n_eval_runs} seed)")
    _eval_multi(run_closed_set_eval, users_seen, "CLOSED-SET", "closedset",
                extra=dict(population_users=seen))

if __name__ == "__main__":
    p = argparse.ArgumentParser(description="Active Auth — pipeline sản phẩm cuối")
    p.add_argument("--data_dir", default="./processed_data")
    p.add_argument("--context", choices=["walking", "all"], default="all")
    p.add_argument("--held_out", default="",
                   help="Danh sách user giữ lại, phẩy ngăn cách, ví dụ user21,user22")
    p.add_argument("--n_eval_runs", type=int, default=5)
    p.add_argument("--quiet", action="store_true")
    a = p.parse_args()
    held = [x.strip() for x in a.held_out.split(",") if x.strip()]
    main(data_dir=a.data_dir, held_out_ids=held,
         n_eval_runs=a.n_eval_runs, verbose=not a.quiet)

## 4. Kiểm chứng đa phương pháp (module benchmark)

In [ ]:
%%writefile method_benchmark.py
"""
method_benchmark.py
====================================================================
KIEM CHUNG da phuong phap SCORING tren backbone ĐA TRAIN cua ban
(khong train lai), tren ca CLOSED-SET (SEEN) va OPEN-SET (HELD-OUT),
dung dung split SEEN/HELD-OUT cua pipeline.

Module nay nạp lai backbone.pt (theo tung context mode), trich embedding
cho moi user, roi so cac phuong phap quyet dinh:

  OWNER VERIFICATION (dung voi deployment):
    cos_mean   : trung binh cosine toi anchor      (BASELINE hien tai)
    cos_knn    : trung binh k cosine lon nhat       (k-NN)
    maha       : -Mahalanobis toi phan bo owner     (Lee 2018)
    cos_znorm  : cos_mean + cohort z-norm           (speaker-verif)

  KNOWN-vs-UNKNOWN (set membership, tham khao — dung LOGIT that cua backbone):
    msp        : Max Softmax Prob
    energy     : Energy score (Liu 2020)
    openmax    : OpenMax / EVT-Weibull (Bendale 2016)

Chi so: pooled AUC + EER voi MOT nguong toan cuc (sat deployment).

Cach dung trong notebook (sau khi pipeline da chay xong, da co backbone.pt):
    from method_benchmark import run_for_mode
    run_for_mode(DATA_DIR, mode='all',     held_out_ids=HELD_OUT_IDS)
    run_for_mode(DATA_DIR, mode='walking', held_out_ids=HELD_OUT_IDS)

(torch & cac module pipeline duoc lazy-import ben trong ham, nen file import
 duoc ca khi chua co GPU.)
"""

from __future__ import annotations
import json
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.special import logsumexp
from scipy.stats import weibull_min
from sklearn.covariance import LedoitWolf
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")

EXCLUDE_USERS  = {"user17", "user11"}   # khop main.py
ANCHOR_COUNT   = 6
ENROLL_FRAC    = 0.4
KNN_K          = 3
ZNORM_COHORT_N = 200
RNG = np.random.default_rng(42)


# ============================================================ SCORERS
def _l2(M):
    return M / (np.linalg.norm(M, axis=-1, keepdims=True) + 1e-9)


def scorer_cos_mean(anchors, cohort=None):
    A = _l2(anchors)
    return lambda Q: (_l2(Q) @ A.T).mean(1)


def scorer_cos_knn(anchors, cohort=None, k=KNN_K):
    A = _l2(anchors)
    def f(Q):
        S = _l2(Q) @ A.T
        kk = min(k, S.shape[1])
        return np.sort(S, axis=1)[:, -kk:].mean(1)
    return f


def scorer_maha(anchors, cohort=None):
    mu = anchors.mean(0)
    base = cohort if (cohort is not None and len(cohort) > anchors.shape[1]) else anchors
    try:
        VI = LedoitWolf().fit(base).precision_
    except Exception:
        VI = np.eye(anchors.shape[1])
    def f(Q):
        d = Q - mu
        m = np.einsum("ij,jk,ik->i", d, VI, d)
        return -np.sqrt(np.clip(m, 0, None))
    return f


def scorer_cos_znorm(anchors, cohort=None):
    base = scorer_cos_mean(anchors)
    if cohort is None or len(cohort) < 5:
        return base
    cs = base(cohort); mu, sd = float(cs.mean()), float(cs.std() + 1e-9)
    return lambda Q: (base(Q) - mu) / sd


VERIF_SCORERS = {
    "cos_mean": scorer_cos_mean, "cos_knn": scorer_cos_knn, "maha": scorer_maha,
    "cos_znorm": scorer_cos_znorm,
}


# ============================================================ METRICS
def eer_from_scores(labels, scores):
    order = np.argsort(-scores); y = labels[order]
    P = y.sum(); Nn = len(y) - P
    if P == 0 or Nn == 0:
        return np.nan
    tp = np.cumsum(y); fp = np.cumsum(1 - y)
    far = fp / Nn; frr = 1 - tp / P
    idx = int(np.nanargmin(np.abs(far - frr)))
    return float((far[idx] + frr[idx]) / 2)


# ============================================================ PROTOCOL (fixed split)
def _enroll_anchors(udata):
    sess = np.array(sorted(set(udata["sess"]),
                           key=lambda x: int(x.split("_")[-1])))
    n_enr = max(1, int(np.ceil(len(sess) * ENROLL_FRAC)))
    enr = set(sess[:n_enr]); test = list(sess[n_enr:])
    Zenr = udata["Z"][np.isin(udata["sess"], list(enr))]
    if len(Zenr) > ANCHOR_COUNT:
        idx = np.linspace(0, len(Zenr) - 1, ANCHOR_COUNT).round().astype(int)
        return Zenr[idx], test
    return Zenr, test


def _session_trials(fn, owner, test_sess, impostors):
    labs, scs = [], []
    for s in test_sess:
        m = owner["sess"] == s
        if m.sum():
            scs.append(float(fn(owner["Z"][m]).mean())); labs.append(1)
    for iu in impostors:
        for s in sorted(set(iu["sess"])):
            m = iu["sess"] == s
            if m.sum():
                scs.append(float(fn(iu["Z"][m]).mean())); labs.append(0)
    return np.array(labs), np.array(scs)


def run_verification(emb, owners, cohort_users, label):
    """owners: list user lam chu may; cohort_users: list user dung cho z-norm/maha.
       Impostor cua moi owner = cac owner CON LAI."""
    cohort = (np.concatenate([emb[u]["Z"] for u in cohort_users])
              if cohort_users else None)
    if cohort is not None and len(cohort) > ZNORM_COHORT_N:
        cohort = cohort[RNG.choice(len(cohort), ZNORM_COHORT_N, replace=False)]

    rows = []
    for mname, make in VERIF_SCORERS.items():
        L_all, S_all = [], []
        for owner in owners:
            anchors, test_sess = _enroll_anchors(emb[owner])
            imp = [emb[u] for u in owners if u != owner]
            if len(anchors) == 0 or not test_sess or not imp:
                continue
            try:
                fn = make(anchors, cohort=cohort)
                l, s = _session_trials(fn, emb[owner], test_sess, imp)
                if len(l):
                    L_all.append(l); S_all.append(s)
            except Exception:
                continue
        if not L_all:
            continue
        L = np.concatenate(L_all); S = np.concatenate(S_all)
        if L.sum() == 0 or (1 - L).sum() == 0:
            continue
        rows.append(dict(method=mname, AUC=roc_auc_score(L, S),
                         EER=eer_from_scores(L, S),
                         n_genuine=int(L.sum()), n_impostor=int((1 - L).sum())))
    df = pd.DataFrame(rows).sort_values("EER").reset_index(drop=True)
    print(f"\n--- OWNER VERIFICATION · {label} ---")
    print(df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    return df


# ============================================================ MEMBERSHIP OOD (logit that)
def run_membership(model, emb, seen, held, get_logits):
    """OpenMax/Energy/MSP tren LOGIT that cua backbone (SEEN classes).
       Phan biet SEEN-test (in) vs HELD (out)."""
    # logit cho windows cua moi user
    def stack(users):
        Z = np.concatenate([emb[u]["Z"] for u in users]) if users else np.empty((0,))
        return Z
    # in = SEEN test sessions; out = HELD (toan bo)
    in_logits, out_logits, tr_logits, tr_y = [], [], [], []
    for ci, u in enumerate(seen):
        sess = sorted(set(emb[u]["sess"]), key=lambda x: int(x.split("_")[-1]))
        n_enr = max(1, int(np.ceil(len(sess) * 0.6)))
        tr_m = np.isin(emb[u]["sess"], sess[:n_enr])
        te_m = np.isin(emb[u]["sess"], sess[n_enr:])
        if tr_m.sum():
            lg = get_logits(emb[u]["Zraw"][tr_m]); tr_logits.append(lg)
            tr_y.append(np.full(len(lg), ci))
        if te_m.sum():
            in_logits.append(get_logits(emb[u]["Zraw"][te_m]))
    for u in held:
        out_logits.append(get_logits(emb[u]["Zraw"]))
    if not in_logits or not out_logits:
        print("   (khong du du lieu cho membership OOD)"); return None
    Ltr = np.concatenate(tr_logits); ytr = np.concatenate(tr_y)
    Lin = np.concatenate(in_logits); Lout = np.concatenate(out_logits)

    def msp(L):
        e = np.exp(L - L.max(1, keepdims=True)); return (e / e.sum(1, keepdims=True)).max(1)
    def energy(L):
        return logsumexp(L, axis=1)
    # OpenMax: Weibull tren khoang cach toi MAV moi lop
    classes = np.unique(ytr); mav, wbl = {}, {}
    for c in classes:
        Ac = Ltr[ytr == c]; mc = Ac.mean(0); mav[c] = mc
        d = np.linalg.norm(Ac - mc, axis=1)
        tail = np.sort(d)[-min(20, len(d)):]
        try:
            wbl[c] = weibull_min.fit(tail, floc=0)
        except Exception:
            wbl[c] = None
    def openmax_in(L):
        out = np.zeros(len(L))
        for i, a in enumerate(L):
            top = np.argsort(-a)[:min(3, len(classes))]; unk = 0.0
            for r, c in enumerate(top):
                if wbl[c] is None:
                    continue
                w = weibull_min.cdf(np.linalg.norm(a - mav[c]), *wbl[c])
                unk += w * a[c] * (r + 1) / len(top)
            out[i] = -unk
        return out

    rows = []
    for name, fn in [("msp", msp), ("energy", energy), ("openmax", openmax_in)]:
        s = np.concatenate([fn(Lin), fn(Lout)])
        y = np.concatenate([np.ones(len(Lin)), np.zeros(len(Lout))])
        rows.append(dict(method=name, AUC=roc_auc_score(y, s)))
    df = pd.DataFrame(rows).sort_values("AUC", ascending=False).reset_index(drop=True)
    print("\n--- KNOWN-vs-UNKNOWN (logit that) — AUC cao = tach known/unknown tot ---")
    print(df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    return df


# ============================================================ DRIVER
def run_for_mode(data_dir, mode="all", held_out_ids=None, do_membership=True):
    """Nap backbone da train cho `mode`, trich embedding, so sanh phuong phap."""
    import torch
    from models import build_backbone
    from dataset import load_users
    from backbone_train import extract_embeddings, DEVICE

    held_out_ids = held_out_ids or []
    exp = Path(f"export_{mode}"); mdl = Path(f"models_{mode}")
    meta = json.load(open(exp / "backbone_metadata.json", encoding="utf-8"))
    seen_trained = meta["users_trained"]; D = int(meta.get("embed_dim", 128))

    print("\n" + "═" * 72)
    print(f"  KIEM CHUNG PHUONG PHAP — context={mode!r}  "
          f"(backbone train tren {len(seen_trained)} user)")
    print("═" * 72)

    backbone = build_backbone("cnn", n_users=len(seen_trained), embed_dim=D).to(DEVICE)
    sd = torch.load(mdl / "backbone.pt", map_location=DEVICE)
    backbone.load_state_dict(sd); backbone.eval()

    users = load_users(Path(data_dir), exclude_users=EXCLUDE_USERS, context_mode=mode)
    seen = [u for u in seen_trained if u in users]
    held = [u for u in held_out_ids if u in users and u not in seen]

    # trich embedding cho moi user (giu ca ban raw cho membership/logit)
    emb = {}
    for u in users:
        Zr = extract_embeddings(backbone, users[u]["X"]).astype(np.float64)
        sess = np.array(["_".join(s.split("_")[-2:])
                         for s in users[u]["session"].astype(str)])
        emb[u] = {"Zraw": Zr, "sess": sess}

    # chuan hoa: fit scaler tren SEEN (tranh ro ri thong tin held-out)
    seenZ = np.concatenate([emb[u]["Zraw"] for u in seen])
    scaler = StandardScaler().fit(seenZ)
    for u in emb:
        Z = scaler.transform(emb[u]["Zraw"])
        emb[u]["Z"] = (Z / (np.linalg.norm(Z, axis=1, keepdims=True) + 1e-9))

    print(f"  SEEN={len(seen)}  HELD-OUT={len(held)}  embed_dim={D}")

    # CLOSED-SET: chu may = SEEN ; cohort = SEEN
    run_verification(emb, owners=seen, cohort_users=seen, label="CLOSED-SET (SEEN)")
    # OPEN-SET: chu may = HELD-OUT (backbone CHUA tung thay) ; cohort = SEEN
    if len(held) >= 2:
        run_verification(emb, owners=held, cohort_users=seen,
                         label="OPEN-SET (HELD-OUT, nguoi moi)")
    else:
        print("\n   (Can >=2 user HELD-OUT de co impostor cho open-set)")

    if do_membership and len(held) >= 1:
        def get_logits(Xraw_emb):
            import torch as T
            with T.no_grad():
                xb = T.from_numpy(Xraw_emb.astype(np.float32)).to(DEVICE)
                # classifier nhan embedding -> logits
                lg = backbone.classifier(xb).cpu().numpy()
            return lg
        try:
            run_membership(backbone, emb, seen, held, get_logits)
        except Exception as e:
            print(f"   (membership OOD bo qua: {e})")

    return emb

## 5. Train backbone + đánh giá open/closed-set (mỗi context mode)

In [ ]:
import time
held_arg = ",".join(HELD_OUT_IDS)
for mode in CONTEXT_MODES:
    print("\n" + "="*72 + f"\n  RUN PIPELINE — context = {mode!r}\n" + "="*72)
    t0 = time.time()
    r = subprocess.run([sys.executable, "main.py",
                        "--data_dir", DATA_DIR,
                        "--context",  mode,
                        "--held_out", held_arg,
                        "--n_eval_runs", str(N_EVAL_RUNS)])
    print(f"  -> {mode}: exit {r.returncode}, {(time.time()-t0)/60:.1f} phút")

## 6. KIỂM CHỨNG — so 7 phương pháp scoring (closed-set + open-set)

In [ ]:
if RUN_BENCHMARK:
    import importlib, method_benchmark
    importlib.reload(method_benchmark)
    from method_benchmark import run_for_mode
    for mode in CONTEXT_MODES:
        try:
            run_for_mode(DATA_DIR, mode=mode, held_out_ids=HELD_OUT_IDS)
        except Exception as e:
            print(f"  [bỏ qua mode={mode!r}] {e}")
    print("\nXong. So sánh bảng EER giữa các phương pháp để chọn cấu hình.")

## 7. (Tùy chọn) Nén artifacts để tải về

In [ ]:
if PACKAGE_ZIP:
    import shutil, os
    for mode in CONTEXT_MODES:
        for d in (f"export_{mode}", f"results_{mode}", f"models_{mode}"):
            if os.path.isdir(d):
                shutil.make_archive(d, "zip", d)
                print("đã nén", d + ".zip")
    try:
        from google.colab import files
        for mode in CONTEXT_MODES:
            for d in (f"results_{mode}", f"export_{mode}"):
                if os.path.exists(d + ".zip"):
                    files.download(d + ".zip")
    except Exception as e:
        print("Tải thủ công các .zip trong panel Files. (", e, ")")